# The carry circuit

What algorithm does the trained model use, and which components implement it?

`digit_addition.ipynb` covers the task, the training setup and the `frac_train` sweep. This
notebook starts from a saved checkpoint so it runs in seconds instead of retraining. Anything
shared (vocabulary, dataset, model construction, the `Trainer`) is imported from `train.py`,
so there is one definition of each and no drift between the code that produced these weights
and the code analysing them.

## Summary of what I found

**Attention routes, the MLP computes.** Two heads fetch operand digits from fixed positions
regardless of the input. Zeroing the MLP destroys every answer position; zeroing all of
attention leaves the leading digit largely intact.

**There is no dedicated carry head.** I expected one head to attend to the units digits more
strongly when a carry occurs. It does not. The routing is identical on carry and non-carry
inputs, and ablating either routing head takes both the units and tens digits to chance.

**The circuit is not stable.** Which heads matter changes over training and differs between
seeds. Every reorganisation after the model converges happens at a loss spike, and weight
decay determines whether the model recovers into a different circuit or back into the same
memorised one.

In [ ]:
import sys

sys.path.append("..")

import einops
import torch as t

from train import (
    DEVICE as device,
    TOKEN_TO_ID,
    VOCABULARY,
    Trainer,
    TrainArgs,
    create_model,
    sum_dataset,
    to_str_tokens,
    to_string,
    to_tokens,
)

t.set_grad_enabled(False)  # analysis only; nothing here trains
print("device:", device)

In [ ]:
from pathlib import Path

CHECKPOINTS = Path("../results/checkpoints")


def load_checkpoint(path, enable_hooks: bool = True):
    """Rebuild the model a checkpoint came from, load its weights, and return it.

    The hook flags are switched on *after* loading, not at training time.
    TransformerLens reads `cfg.use_attn_result` / `cfg.use_split_qkv_input` inside
    `forward` rather than at construction, so the same weights work either way --
    and leaving them on during training costs ~140x throughput.
    """
    ckpt = t.load(path, weights_only=False, map_location=device)

    # the checkpoint stores asdict(args); seq_len and run_name are derived in
    # __post_init__ so they are not fields, and device is whatever the cluster used
    fields = TrainArgs.__dataclass_fields__
    saved = {k: v for k, v in ckpt["config"].items() if k in fields}
    saved["device"] = str(device)
    args_ck = TrainArgs(**saved)

    model = create_model(args_ck)

    # `attn.mask` and `attn.IGNORE` are derived buffers (tril of n_ctx, and -inf),
    # not learned weights. TransformerLens versions disagree on their saved shape,
    # so drop them and let the fresh model keep its own. strict=False is required
    # once keys are removed, so check explicitly that nothing real went missing.
    derived = (".mask", ".IGNORE")
    sd = {k: v for k, v in ckpt["state_dict"].items() if not k.endswith(derived)}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    real_missing = [k for k in missing if not k.endswith(derived)]
    assert not real_missing, f"missing learned weights: {real_missing}"
    assert not unexpected, f"unexpected keys: {unexpected}"

    model.eval()

    if enable_hooks:
        model.cfg.use_attn_result = True
        model.cfg.use_split_qkv_input = True

    return model, args_ck, ckpt


print("available checkpoints:")
for f in sorted(CHECKPOINTS.glob("*_best.pt")):
    print(" ", f.name)

In [ ]:
# Verify before analysing anything. load_state_dict will happily accept a model
# built from a slightly different config and give plausible-looking garbage.

CKPT = CHECKPOINTS / "L1_wd1.0_ft0.015_s2_best.pt"

model, args_ck, ckpt = load_checkpoint(CKPT)
train_ck, test_ck = sum_dataset(
    num_digits=args_ck.num_digits, seed=args_ck.seed, frac_train=args_ck.frac_train
)

trainer_ck = Trainer(model, args_ck)
with t.inference_mode():
    loss, digit_acc, seq_acc = trainer_ck.validation_step(test_ck)

print(f"checkpoint from step {ckpt['step']}")
print(f"  stored test_seq_accuracy : {ckpt['test_seq_accuracy']:.4f}")
print(f"  recomputed               : {seq_acc:.4f}")
print(f"  per-digit                : {digit_acc:.4f}   loss {loss:.4f}")

assert abs(seq_acc - ckpt["test_seq_accuracy"]) < 1e-3, (
    "recomputed accuracy does not match the checkpoint - the model you just built "
    "is not the model that was trained"
)
print("\nOK - safe to analyse this model")

## Attention patterns

The tens digit is the only output that needs carry information, so it is where a carry
mechanism has to show up. Given the layout below, the tens digit `c2` is predicted from
query position 6.

```
position   0    1    2    3    4    5    6    7    8
token      a1   a2   +    b1   b2   =    c1   c2   c3
```

To determine whether `a2 + b2 >= 10`, something has to read positions 1 and 4. I compared a
carry example against a non-carry one, expecting a head to attend to those positions more
strongly on the carry input.

**It does not.** Head 3 does attend to `a2` and `b2` (roughly 0.30 each), and head 1 attends
to the tens operands `a1` and `b1` (0.39 and 0.42), but both do so identically on carry and
non-carry inputs. Every difference in the table below is under 0.05 on a budget of 1.0.

The reading is that attention here is positional routing rather than computation. The tens
output always needs both digit pairs, because you cannot know whether a carry occurred until
you have looked at the units, so the heads fetch them unconditionally. The carry itself is
computed downstream. That is the one-layer version of the split Kruthoff (arXiv:2401.07993)
describes across two layers, where attention decides which positions matter and the MLP does
the carrying.

Head 0 is worth noting: its attention is 0.143 at every visible position, which is exactly
1/7. Its scores are constant, so it is passing a uniform average of the sequence.

In [ ]:
# Contrast pair: same structure, different carry behaviour.
#   12+34=640   2+4=6,  1+3=4      no carry anywhere
#   49+97=641   9+7=16, 4+9+1=14   carry at both positions
# Note the answer is zero-padded to num_digits+1 and then reversed, so 46 -> "046" -> "640".
PAIR = ["12+34=640", "49+97=641"]
toks = to_tokens(PAIR)

logits, cache = model.run_with_cache(toks)

print(f"tokens {tuple(toks.shape)}   logits {tuple(logits.shape)}")
for s, row in zip(PAIR, to_str_tokens(toks)):
    print(f"  {s}  ->  {row}")

print("""
position   0    1    2    3    4    5    6    7    8
token      a1   a2   +    b1   b2   =    c1   c2   c3

logits at position i predict token i+1, so the tens digit c2 is
predicted from query position 6 -- the only output needing a carry.
""")

print("cache entries:")
for k in cache.keys():
    print(f"  {k:<42s} {tuple(cache[k].shape)}")

In [ ]:
pattern = cache["pattern", 0]          # (batch, head, query_pos, key_pos)
QUERY = 6                             # logit here predicts c2, the tens digit

rows = pattern[:, :, QUERY, :].cpu()  # (batch, head, key_pos)
LABELS = ["a1", "a2", "+", "b1", "b2", "=", "c1", "c2", "c3"]
header = "        " + "".join(f"{l:>7}" for l in LABELS)

for b, s in enumerate(PAIR):
    carry = "carry" if b else "no carry"
    print(f"\n{s}   ({carry})   attention at query {QUERY}")
    print(header)
    for h in range(model.cfg.n_heads):
        print(f"  head{h}  " + "".join(f"{v:7.3f}" for v in rows[b, h]))

print("\n\ndifference: carry minus no-carry")
print(header)
for h in range(model.cfg.n_heads):
    print(f"  head{h}  " + "".join(f"{v:+7.3f}" for v in (rows[1] - rows[0])[h]))

# causal mask means query 6 can only see keys 0-6, so the last two columns are 0
# each row sums to 1 across visible keys -- "strong" is relative to that budget

## Ablation

Attention patterns show what a head looks at. They cannot show whether the model uses it.
Zeroing a head's output at `hook_z` and measuring the damage can.

Accuracy is broken out per answer position, because the interesting prediction is asymmetric:
the units digit needs no carry, so a component supplying only carry information should break
the tens digit and leave the units alone.

Results for the `ft0.015 s2` checkpoint (step 71,600, baseline 0.975 whole-sum):

| ablated | units | tens | hundreds | whole sum |
|---|---|---|---|---|
| none | 0.990 | 0.985 | 1.000 | 0.975 |
| head 0 | 0.990 | 0.985 | 1.000 | 0.975 |
| head 1 | 0.101 | 0.100 | 1.000 | 0.010 |
| head 2 | 0.283 | 0.578 | 0.758 | 0.155 |
| head 3 | 0.130 | 0.127 | 0.635 | 0.012 |
| all attention | 0.100 | 0.100 | 0.708 | 0.009 |
| MLP | 0.153 | 0.091 | 0.069 | 0.000 |

Three things follow.

Head 0 is inert. Ablating it changes nothing to three decimals, confirming what its uniform
attention suggested. That is also the control: if removing a component that looks inert had
broken something, the ablation setup would be wrong.

Heads 1 and 3 are both required for both digits, so the carry is not separable from the rest
of the addition. My prediction that head 3 would break the tens digit while sparing the units
was wrong.

The leading digit behaves differently from the other two. It survives head 1 ablation
completely and sits at 0.708 with all attention removed. That is not evidence of a separate
mechanism: position 7 still sees its own token `c2` without any attention, and because these
sums cluster near 99, `c3` is 75.0% predictable from `c2` alone (tens digit 0 implies a
hundreds digit of 1, tens digit 9 implies 0). So 0.708 is *below* the no-attention lookup
ceiling, not above it. The one real observation here is that zeroing the MLP takes the
leading digit to 0.069, far below the 0.505 majority-class rate.

In [ ]:
from functools import partial

# hook_z is always populated; use_attn_result / use_split_qkv_input are not needed here
# and materialise large per-head tensors, so switch them off for the sweep over the
# full test set and back on afterwards for pattern work.
model.cfg.use_attn_result = False
model.cfg.use_split_qkv_input = False

ND = args_ck.num_digits
POS_NAMES = ["units c1", "tens c2", "hundreds c3"]
test_toks = test_ck.to(device)


def zero_head(z, hook, head):
    """z: (batch, pos, head, d_head)"""
    z[:, :, head, :] = 0.0
    return z


def zero_all(act, hook):
    act[:] = 0.0
    return act


def accuracy(fwd_hooks=()):
    """Per-answer-position accuracy, and whole-sum accuracy."""
    logits = model.run_with_hooks(test_toks, fwd_hooks=list(fwd_hooks))
    logits = logits[:, -(ND + 2) : -1]      # positions predicting c1, c2, c3
    target = test_toks[:, -(ND + 1) :]
    correct = logits.argmax(-1) == target   # (batch, 3)
    return correct.float().mean(0).cpu(), correct.all(-1).float().mean().item()


runs = [("none (baseline)", ())]
runs += [
    (f"head {h}", ((f"blocks.{0}.attn.hook_z", partial(zero_head, head=h)),))
    for h in range(model.cfg.n_heads)
]
runs += [("all attn", (("blocks.0.hook_attn_out", zero_all),)),
         ("mlp", (("blocks.0.hook_mlp_out", zero_all),))]

print(f"{'ablated':>16}  " + "".join(f"{n:>13}" for n in POS_NAMES) + f"{'whole sum':>12}")
print("-" * 16 + "  " + "-" * (13 * len(POS_NAMES) + 12))
for label, hooks in runs:
    pos, seq = accuracy(hooks)
    print(f"{label:>16}  " + "".join(f"{v:13.3f}" for v in pos) + f"{seq:12.3f}")

model.cfg.use_attn_result = True
model.cfg.use_split_qkv_input = True

## When does the circuit form?

The single-checkpoint ablation describes the finished model. The 50 saved snapshots show how
it got there, which is the question Nanda et al.'s progress-measures work is built around.

Test accuracy reaches 0.93 around step 24,000 and ends at 0.95, and at this 2,000-step
sampling it looks flat for the remaining 75,000 steps. It is not: the underlying history,
logged every 100 steps, drops as far as 0.385 at step 58,200 before recovering. The
checkpoint grid is too coarse to see the collapses, only the states between them.

The circuit underneath does not settle either. Head 1 is essential, does nothing between
steps 28,000 and 42,000, then becomes essential again. Head 0 flips three times. By step
82,000 the model has moved to a solution using heads 0 and 1, having previously used 1, 2
and 3. Any statement about which head does what is a statement about one checkpoint.

In [ ]:
import json
import re

STEP_RE = re.compile(r"step(\d+)")
STEP_CKPTS = sorted(CHECKPOINTS.glob("*_step*.pt"),
                    key=lambda f: int(STEP_RE.search(f.name).group(1)))
print(f"{len(STEP_CKPTS)} checkpoints, steps "
      f"{int(STEP_RE.search(STEP_CKPTS[0].name).group(1))}-"
      f"{int(STEP_RE.search(STEP_CKPTS[-1].name).group(1))}")

formation = []
for f in STEP_CKPTS:
    m, a_ck, ck = load_checkpoint(f, enable_hooks=False)
    globals()["model"] = m                       # accuracy() closes over `model`
    base_pos, base_seq = accuracy()
    row = {"step": ck["step"], "baseline": base_seq,
           "baseline_pos": base_pos.tolist(), "heads": {}}
    for h in range(m.cfg.n_heads):
        _, seq = accuracy((("blocks.0.attn.hook_z", partial(zero_head, head=h)),))
        row["heads"][h] = seq
    formation.append(row)
    print(f"  step {ck['step']:>6}  base {base_seq:.3f}  "
          + "  ".join(f"h{h} {row['heads'][h]:.3f}" for h in range(m.cfg.n_heads)))

with open("../results/circuit_formation.json", "w") as fh:
    json.dump(formation, fh)
print("\nsaved ../results/circuit_formation.json")

## Is the circuit the same across seeds?

If which-head-does-what changes over training within one run, the obvious next question is
whether it is even consistent between runs. Four checkpoints, three of them differing only in
seed.

In each model one head matters much less than the other three, but which one depends on the
seed. Damage is the drop in whole-sum accuracy when the head is zeroed:

| checkpoint | h0 | h1 | h2 | h3 | least load-bearing |
|---|---|---|---|---|---|
| `ft0.02 s0` | 0.97 | 0.98 | 0.00 | 0.73 | head 2 |
| `ft0.02 s1` | 0.99 | 0.16 | 0.99 | 0.97 | head 1 |
| `ft0.02 s2` | 0.97 | 0.19 | 0.90 | 0.66 | head 1 |
| `ft0.015 s2` | 0.00 | 0.97 | 0.82 | 0.96 | head 0 |

Only two of the four have a genuinely free head (damage < 0.05). In `ft0.02 s1` and `s2` the
weakest head still costs 0.16 and 0.19, so "three heads suffice" overstates it — what holds
across seeds is that the four heads are never equally loaded, and nothing privileges a
particular head for a particular job.

In [ ]:
CKPT_SET = [
    "L1_wd1.0_ft0.02_s0_best.pt",
    "L1_wd1.0_ft0.02_s1_best.pt",
    "L1_wd1.0_ft0.02_s2_best.pt",
    "L1_wd1.0_ft0.015_s2_best.pt",
]

universality = []
print(f"{'checkpoint':>22}{'step':>8}{'base':>8}" + "".join(f"{'h'+str(h):>8}" for h in range(4)))
for name in CKPT_SET:
    m, a, ck = load_checkpoint(CHECKPOINTS / name, enable_hooks=False)
    globals()["model"] = m
    _, test = sum_dataset(num_digits=a.num_digits, seed=a.seed, frac_train=a.frac_train)
    globals()["test_ck"] = test.to(device)
    globals()["test_toks"] = globals()["test_ck"]

    _, base = accuracy()
    dmg = [base - accuracy(((f"blocks.0.attn.hook_z", partial(zero_head, head=h)),))[1]
           for h in range(4)]
    universality.append({"ckpt": name, "step": ck["step"], "base": base, "damage": dmg})
    print(f"{name.replace('L1_wd1.0_', ''):>22}{ck['step']:>8}{base:>8.3f}"
          + "".join(f"{d:>8.3f}" for d in dmg))

print("\nload-bearing (damage > 0.5):")
for r in universality:
    ess = [h for h, d in enumerate(r["damage"]) if d > 0.5]
    spare = [h for h, d in enumerate(r["damage"]) if d < 0.05]
    print(f"  {r['ckpt'].replace('L1_wd1.0_', ''):>22}  essential {ess}  redundant {spare}")

with open("../results/universality.json", "w") as fh:
    json.dump(universality, fh)

## Reorganisation follows the loss spikes

Splitting the head swaps by phase, taking the accuracy plateau at around step 30,000:

- **7 swaps before the plateau.** These are the circuit forming while accuracy is still
  climbing, and they do not line up with spikes.
- **7 swaps after the plateau.** All 7 fall within 2,500 steps of a loss spike, which is the
  resolution limit given checkpoints every 2,000 steps.

So reorganisation of an already-working circuit is not gradual drift. It happens at discrete
moments, and those moments are the loss spikes.

**What causes the spikes is not weight decay.** I assumed it was, on the grounds that decay
keeps shrinking weights with nothing to oppose it once the loss gradient vanishes. The
`weight_decay=0` control refutes that: it has *nine* spikes to this run's six. The spikes are
the Adam instability at vanishing loss, where the second-moment estimate collapses along with
the gradients and the next real gradient produces an enormous step. That happens with or
without decay.

What weight decay changes is what the model re-converges *to*. Without it, the model returns
to the same memorised solution and test accuracy stays at 0.08 for the entire run. With it,
each recovery lands on a different set of heads, and test accuracy climbs to 0.975.

A spike has to be defined relatively, not absolutely: this run sits at about 5e-6 train loss,
so any fixed threshold catches everything or nothing. A jump of more than 10x the previous
value finds six.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

RUN = "L1_wd1.0_ft0.015_s2"
hist = json.load(open(f"../results/sweep/history_{RUN}.json"))["history"]
hs, tl = np.array(hist["step"]), np.array(hist["train_loss"])

F = json.load(open("../results/circuit_formation.json"))
steps = np.array([r["step"] for r in F])
base = np.array([r["baseline"] for r in F])
dmg = {h: base - np.array([r["heads"][str(h)] for r in F]) for h in range(4)}

# absolute thresholds are meaningless at ~5e-6 loss; a spike is a relative jump
ratio = tl[1:] / np.maximum(tl[:-1], 1e-12)
onsets = hs[np.where(ratio > 10)[0] + 1]
onsets = onsets[onsets > 2000]

PLATEAU, WIN = 30_000, 2_500        # WIN = checkpoint spacing, the resolution limit
swaps = [(steps[i], h, "on" if dmg[h][i] >= 0.5 else "off")
         for h in range(4) for i in range(1, len(steps))
         if (dmg[h][i - 1] < 0.5) != (dmg[h][i] < 0.5)]
swaps.sort()

print(f"loss spikes: {onsets.tolist()}\n")
for s, h, dirn in swaps:
    near = onsets[np.abs(onsets - s) <= WIN]
    print(f"  step {s:>6}  head {h} {dirn:>3}" + (f"   spike at {near.tolist()}" if len(near) else ""))

reorg = [s for s, _, _ in swaps if s >= PLATEAU]
events = sorted(set(reorg))                     # flips at the same step are one event
matched_ev = sum(1 for s in events if np.any(np.abs(onsets - s) <= WIN))

# a hit rate is meaningless without the base rate: what fraction of the checkpoint grid
# is within WIN of *some* spike anyway?
grid = steps[steps >= PLATEAU]
base_rate = np.mean([np.any(np.abs(onsets - s) <= WIN) for s in grid])

print(f"\nbefore step {PLATEAU} (forming):        {len(swaps) - len(reorg)} swaps")
print(f"after  step {PLATEAU} (reorganisation): {len(reorg)} head flips at "
      f"{len(events)} distinct steps, {matched_ev} of them at a loss spike")
print(f"base rate: {base_rate:.0%} of post-{PLATEAU} checkpoints are within {WIN} of a spike "
      f"anyway, so {matched_ev}/{len(events)} by chance has p = {base_rate**len(events):.4f}")

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2dd"
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                             gridspec_kw={"height_ratios": [1, 1.4]})
fig.patch.set_facecolor(SURFACE)
for ax in (a1, a2):
    ax.set_facecolor(SURFACE); ax.grid(True, color=GRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): ax.spines[sp].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    for o in onsets:
        ax.axvline(o, color="#b0aea6", linewidth=1.1, zorder=1)

a1.plot(hs, tl, color=INK2, linewidth=1.4, zorder=3)
a1.set_yscale("log"); a1.set_ylabel("train loss", color=INK2, fontsize=10)
a1.set_title("Head swaps coincide with training-loss spikes",
             color=INK, fontsize=13, loc="left", pad=12)
a1.annotate("grey lines = loss spikes", xy=(0.985, 0.9), xycoords="axes fraction",
            ha="right", fontsize=8, color=INK2)

for h in range(4):
    a2.plot(steps, dmg[h], color=COL[h], linewidth=2, zorder=3, label=f"head {h}")
    a2.annotate(f"h{h}", xy=(steps[-1], dmg[h][-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color=INK2)
a2.axhline(0.5, color=INK2, linewidth=0.9, linestyle=":", zorder=2)
a2.set_ylabel("damage when ablated", color=INK2, fontsize=10)
a2.set_xlabel("optimizer step", color=INK2, fontsize=10)
a2.set_ylim(-0.08, 1.05)
a2.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="center left")
fig.tight_layout()
fig.savefig("../results/spike_alignment.png", dpi=200, facecolor=SURFACE)
plt.show()